In [1]:
import numpy as np 
import pandas as pd 

import mlflow
import mlflow.sklearn
import dagshub

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import os 

/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_owner='anni0955', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/anni0955/mlops-mini-project.mlflow')

Accessing as anni0955

Initialized MLflow to track repo "anni0955/mlops-mini-project"

Repository anni0955/mlops-mini-project initialized!

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def lemmatization(text):
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return ' '.join(text)

def remove_stop_words(text):
    text = [word for word in str(text).split() if word not in stop_words]
    return ' '.join(text)

def remove_numbers(text):
    text = ''.join(char for char in text if not char.isdigit())
    return text

def lowercase(text):
    text = text.split()
    text = [word.lower() for word in text]
    return ' '.join(text)

def remove_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(':', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lowercase)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_punctuations)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'error in text_normalization: {e}')
        raise 

In [5]:
df = normalize_text(df)

In [6]:
x = df['sentiment'].isin(['happiness', 'sadness'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})
df.head()

/tmp/ipykernel_81699/2161997397.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})


,tweet_id,sentiment,content
1,1956967666,0,layin n bed headache ughhhh waitin call
2,1956967696,0,funeral ceremony gloomy friday
6,1956968487,0,sleep im thinking old friend want married damn...
8,1956969035,0,charviray charlene love miss
9,1956969172,0,kelcouch sorry least friday


In [8]:
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(df['content'])
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=42) 

In [9]:
mlflow.set_experiment('LR Hyperparameter Tuning')

2026/03/28 16:15:35 INFO mlflow.tracking.fluent: Experiment with name 'LR Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e69d66645dc54968b0ee6721f0724030', creation_time=1774694738927, experiment_id='2', last_update_time=1774694738927, lifecycle_stage='active', name='LR Hyperparameter Tuning', tags={}, workspace='default'>

In [10]:
param_grid = {
    'C': [.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

In [11]:
from sklearn.model_selection import GridSearchCV

In [13]:
with mlflow.start_run():
    grid_search = GridSearchCV(LogisticRegression(), param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(x_train, y_train)

    for params, mean_score, std_score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f'LR wuith params: {params}', nested=True):
            model=LogisticRegression(**params)
            model.fit(x_train, y_train)

            y_pred = model.predict(x_test)

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)

            mlflow.log_metric('accuracy', accuracy)
            mlflow.log_metric('precision', precision)
            mlflow.log_metric('recall', recall)
            mlflow.log_metric('f1', f1)
            mlflow.log_metric('std_cv_score', std_score)
            mlflow.log_metric('mean_cv_score', mean_score)

            mlflow.sklearn.log_model(model, "model")

            print(f'mean cv score: {mean_score}, std_cv_score: {std_score}')
            print(f'accuracy: {accuracy}')
            print(f'precision: {precision}')
            print(f'recall: {recall}')
            print(f'mean cv f1: {f1}')

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric('best_f1_score', best_score)

    print(f'best params: {best_params}')
    print(f'best f1_score: {best_score}')

/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to 

mean cv score: 0.6972094030635826, std_cv_score: 0.013051137708852535
accuracy: 0.7402409638554217
precision: 0.7860576923076923
recall: 0.6443349753694582
mean cv f1: 0.7081754195993503
🏃 View run LR wuith params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/c9795f8c8fc54b87abc0068397bf5d16
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2


/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026/03/28 16:36:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:36:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mean cv score: 0.7798550139794358, std_cv_score: 0.01262516306175453
accuracy: 0.7860240963855422
precision: 0.7790811339198436
recall: 0.7852216748768472
mean cv f1: 0.7821393523061825
🏃 View run LR wuith params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/b19d1e9f8bde44398eeb93f63e968aa4
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2


/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/03/28 16:37:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:37:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on 

...

mean cv score: 0.7845866566948729, std_cv_score: 0.005720573360336938
accuracy: 0.8
precision: 0.7890173410404624
recall: 0.8068965517241379
mean cv f1: 0.7978567949342426
🏃 View run LR wuith params: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/13b5413e4bf24ef5813955cd69652123
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2


/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/03/28 16:38:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:38:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on 

mean cv score: 0.7664556374250394, std_cv_score: 0.009919738784378983
accuracy: 0.7797590361445783
precision: 0.7746062992125984
recall: 0.7753694581280788
mean cv f1: 0.7749876907927129
🏃 View run LR wuith params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/45cc040575904795862d1321269b5853
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2


/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026/03/28 16:38:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:38:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mean cv score: 0.7737752775141773, std_cv_score: 0.005127854306640501
accuracy: 0.7816867469879518
precision: 0.7701923076923077
recall: 0.7891625615763547
mean cv f1: 0.7795620437956204
🏃 View run LR wuith params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/81b3e9101b384309871a3008e253d624
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2
best params: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
best f1_score: 0.7845866566948729
🏃 View run powerful-dolphin-771 at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2/runs/8e83719b933e4d708b7b43d8f713b4f0
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/2
